In [2]:
import numpy as np
import pandas as pd
import os
import librosa
import tkinter as tk
from tkinter import filedialog, messagebox, Label, Button
from tensorflow.keras.models import load_model
from sklearn.preprocessing import LabelEncoder

# Load the trained model
model = load_model('E:/Users/Sumit/Downloads/UrbanSound8K/UrbanSound8K/saved_models_CNN/audio_classification_CNN.keras')

# LabelEncoder instance with classes (replace with your actual class labels)
class_names = ["air_conditioner", "car_horn", "children_playing", "dog_bark", "drilling", 
               "engine_idling", "gun_shot", "jackhammer", "siren", "street_music"]

def features_extractor(file):
    # Extract MFCC features for the audio file
    audio, sample_rate = librosa.load(file, res_type='kaiser_fast')
    mfccs = librosa.feature.mfcc(y=audio, sr=sample_rate, n_mfcc=40)
    mfccs_scaled = np.mean(mfccs.T, axis=0)
    return mfccs_scaled

# Define GUI Application
class AudioClassifierApp:
    def __init__(self, root):
        self.root = root
        self.root.title("Acoustic Scene Classification")
        self.root.geometry("400x300")

        # Label
        self.label = Label(root, text="Acoustic Scene Classifier using CNN", font=("Helvetica", 14))
        self.label.pack(pady=10)

        # Buttons for upload and predict
        self.upload_button = Button(root, text="Upload Audio File", command=self.upload_file)
        self.upload_button.pack(pady=5)
        
        self.predict_button = Button(root, text="Predict Class", command=self.predict_class, state="disabled")
        self.predict_button.pack(pady=5)

        # Result label
        self.result_label = Label(root, text="", font=("Helvetica", 12), fg="blue")
        self.result_label.pack(pady=10)

        # Initialize variables
        self.file_path = None

    def upload_file(self):
        # Open file dialog to upload file
        self.file_path = filedialog.askopenfilename(filetypes=[("Audio Files", "*.wav *.mp3")])
        if self.file_path:
            self.result_label.config(text="File loaded successfully!")
            self.predict_button.config(state="normal")
        else:
            self.result_label.config(text="No file selected.")

    def predict_class(self):
        if self.file_path:
            # Extract features from the audio file
            feature = features_extractor(self.file_path)
            feature = feature.reshape(1, 40, 1, 1)  # Reshape as expected by CNN model

            # Predict class
            predictions = model.predict(feature)
            predicted_class_index = np.argmax(predictions, axis=1)[0]
            prediction_label = class_names[predicted_class_index]

            # Display the result
            self.result_label.config(text=f"Predicted class: {prediction_label}")
        else:
            messagebox.showwarning("Warning", "Please upload an audio file first.")

# Main
if __name__ == "__main__":
    root = tk.Tk()
    app = AudioClassifierApp(root)
    root.mainloop()


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 286ms/step
